# WF2020 Low-Exposure Rally Floor Target v1

Purpose: convert the `wf2020_failed_fold_rally_gap_attribution_v1` finding into a training-safe fold-local target contract. The target is diagnostic only and does not promote an actor.

Baselines:
- Canonical: `v19reg_v22_budget_usage_bull_guard_v1`
- Direct comparison: `return_preserving_bull_participation_three_fold_checkpoint_metric_replay_v1`
- Source target: `wf2020_return_preserving_bull_participation_target_v1`

Hypothesis: raising low-exposure, non-downtrend, positive-momentum training rows in `wf_2020_train` to a 0.30 rebound-consumed target floor can improve `wf_2020` return without touching the already passing fold targets.


In [1]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import json
import math
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'AGENTS.md').exists() and (candidate / 'runtime').exists():
            return candidate
    raise RuntimeError('repo root not found')

ROOT=find_repo_root(Path.cwd())
if str(ROOT/'src') not in sys.path:
    sys.path.insert(0,str(ROOT/'src'))
from soxl_quant.policy.policy_builder import state_objective_target_position_from_row

EXPERIMENT='wf2020_low_exposure_rally_floor_target_v1'
CARD=ROOT/'runtime/model_selection/card'/EXPERIMENT
CARD.mkdir(parents=True,exist_ok=True)
SOURCE_CARD=ROOT/'runtime/model_selection/card/wf2020_return_preserving_bull_participation_target_v1'
SOURCE_CONTRACT=SOURCE_CARD/'return_preserving_bull_participation_three_fold_target_contract.csv'
SOURCE_SUMMARY=SOURCE_CARD/'summary.json'
ATTRIBUTION_SUMMARY=ROOT/'runtime/model_selection/card/wf2020_failed_fold_rally_gap_attribution_v1/summary.json'

FLOOR=0.30
MOM5_THRESHOLD=-0.02
TARGET_FOLD='wf_2020_train'

SIDE_CARS={
 'target_contract': CARD/'low_exposure_rally_floor_three_fold_target_contract.csv',
 'target_audit': CARD/'target_audit.csv',
 'changed_rows': CARD/'changed_rows.csv',
 'selection': CARD/'selection.json',
 'summary': CARD/'summary.json',
 'conclusion': CARD/'conclusion.md',
}

def rel(path: Path)->str:
    return str(path.relative_to(ROOT))

def json_safe(value):
    if isinstance(value, dict): return {str(k): json_safe(v) for k,v in value.items()}
    if isinstance(value,(list,tuple)): return [json_safe(v) for v in value]
    if isinstance(value,(np.integer,)): return int(value)
    if isinstance(value,(np.bool_,)): return bool(value)
    if isinstance(value,(np.floating,)): value=float(value)
    if isinstance(value,float): return value if math.isfinite(value) else None
    if isinstance(value,pd.Timestamp): return value.isoformat()
    if isinstance(value,Path): return str(value)
    return value

def write_json(path: Path, payload: dict)->None:
    path.write_text(json.dumps(json_safe(payload),indent=2,sort_keys=True,allow_nan=False)+'\n',encoding='utf-8')

def bool_series(frame: pd.DataFrame, col: str)->pd.Series:
    if col not in frame.columns:
        return pd.Series(False,index=frame.index)
    s=frame[col]
    if s.dtype==bool:
        return s.fillna(False)
    if pd.api.types.is_numeric_dtype(s):
        return s.fillna(0).astype(float).ne(0)
    return s.fillna('').astype(str).str.lower().str.strip().isin({'1','true','yes','y'})

def consumed_target(frame: pd.DataFrame)->pd.Series:
    return pd.Series([state_objective_target_position_from_row(row) for row in frame.to_dict('records')], index=frame.index, dtype='float64')

print('card', rel(CARD))
print('source', rel(SOURCE_CONTRACT))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


card runtime/model_selection/card/wf2020_low_exposure_rally_floor_target_v1
source runtime/model_selection/card/wf2020_return_preserving_bull_participation_target_v1/return_preserving_bull_participation_three_fold_target_contract.csv


In [2]:
source_summary=json.loads(SOURCE_SUMMARY.read_text())
attribution=json.loads(ATTRIBUTION_SUMMARY.read_text())
source=pd.read_csv(SOURCE_CONTRACT, parse_dates=['datetime','train_start','train_end','eval_start','eval_end'])
for col in ['teacher_position','risk_target_position','rebound_target_position','return_preserving_target_position','soxl_mom5_prev','target_delta_vs_source','objective_weight']:
    if col in source.columns:
        source[col]=pd.to_numeric(source[col],errors='coerce')
source['source_consumed_target_position']=consumed_target(source)
print('source rows', len(source), 'folds', sorted(source.fold.astype(str).unique()))
print('attribution decision', attribution.get('decision'))
print('source projected mismatch rows', source_summary.get('projected_mismatch_rows'))


source rows 3731 folds ['wf_2020_train', 'wf_2022_train', 'wf_2023_2024_train']
attribution decision wf2020_low_exposure_rally_gap_requires_training_safe_floor_target_no_promotion
source projected mismatch rows 0


In [3]:
out=source.sort_values(['fold','datetime']).reset_index(drop=True).copy()
train_mask=out['datetime'].between(out['train_start'], out['train_end'])
base_target=pd.to_numeric(out['return_preserving_target_position'], errors='coerce').fillna(out['source_consumed_target_position']).astype(float)
non_downtrend=~bool_series(out,'is_downtrend_state')
low_exposure=base_target < FLOOR
positive_mom=out['soxl_mom5_prev'].fillna(0.0) >= MOM5_THRESHOLD
fold_mask=out['fold'].astype(str).eq(TARGET_FOLD)
change_mask=train_mask & fold_mask & non_downtrend & low_exposure & positive_mom

out['source_return_preserving_target_position']=base_target
out['low_exposure_rally_floor_state_v1']=change_mask.astype(bool)
out.loc[change_mask,'rebound_target_position']=np.maximum(pd.to_numeric(out.loc[change_mask,'rebound_target_position'],errors='coerce'), FLOOR)
out.loc[change_mask,'return_preserving_target_position']=np.maximum(base_target.loc[change_mask], FLOOR)
out.loc[change_mask,'is_rebound_cluster']=True
out.loc[change_mask,'is_downtrend_state']=False
if 'drawdown_control_state' in out.columns:
    out.loc[change_mask,'drawdown_control_state']=False
out.loc[change_mask,'objective_weight']=np.maximum(pd.to_numeric(out.loc[change_mask,'objective_weight'],errors='coerce').fillna(1.0), 7.0)
out['target_semantics_revision']='wf2020_low_exposure_rally_floor_v1'
out.loc[change_mask,'target_reason']='low_exposure_positive_mom_rebound_floor'
out['consumed_state_objective_target_position']=consumed_target(out)
out['target_diff_vs_consumed']=out['return_preserving_target_position'].astype(float)-out['consumed_state_objective_target_position'].astype(float)
out['target_diff_vs_consumed_abs']=out['target_diff_vs_consumed'].abs()
out['consumed_target_mismatch']=out['target_diff_vs_consumed_abs'].gt(1e-9)
out['target_delta_vs_source_floor_v1']=out['return_preserving_target_position'].astype(float)-out['source_return_preserving_target_position'].astype(float)
out['eval_leakage_row']=out['datetime'].between(out['eval_start'],out['eval_end'])
out.to_csv(SIDE_CARS['target_contract'],index=False)

train=out[out['datetime'].between(out['train_start'],out['train_end'])].copy()
rows=[]
for fold,g in train.groupby('fold',sort=True):
    rows.append({
        'fold': fold,
        'train_rows': int(len(g)),
        'changed_rows': int(g['low_exposure_rally_floor_state_v1'].sum()),
        'mismatch_rows': int(g['consumed_target_mismatch'].sum()),
        'target_delta_nonzero_rows': int(g['target_delta_vs_source_floor_v1'].abs().gt(1e-9).sum()),
        'mean_source_target': float(g['source_return_preserving_target_position'].mean()),
        'mean_target': float(g['return_preserving_target_position'].mean()),
        'rebound_rows': int(bool_series(g,'is_rebound_cluster').sum()),
        'downtrend_rows': int(bool_series(g,'is_downtrend_state').sum()),
        'eval_leakage_rows': int(g['eval_leakage_row'].sum()),
    })
audit=pd.DataFrame(rows)
audit.to_csv(SIDE_CARS['target_audit'],index=False)
changed_cols=['datetime','fold','teacher_position','source_return_preserving_target_position','return_preserving_target_position','consumed_state_objective_target_position','target_delta_vs_source_floor_v1','soxl_mom5_prev','is_rebound_cluster','is_downtrend_state','drawdown_control_state','target_reason']
out.loc[out['low_exposure_rally_floor_state_v1'], changed_cols].to_csv(SIDE_CARS['changed_rows'],index=False)
display(audit)
print('changed rows', int(change_mask.sum()))
assert int(audit['eval_leakage_rows'].sum())==0
assert int(audit['mismatch_rows'].sum())==0


,fold,train_rows,changed_rows,mismatch_rows,target_delta_nonzero_rows,mean_source_target,mean_target,rebound_rows,downtrend_rows,eval_leakage_rows
0,wf_2020_train,2470,452,0,452,0.304600,0.335576,737,897,0
1,wf_2022_train,505,0,0,0,0.348817,0.348817,66,234,0
2,wf_2023_2024_train,756,0,0,0,0.277530,0.277530,66,478,0


changed rows 452


In [4]:
changed_rows=int(out['low_exposure_rally_floor_state_v1'].sum())
selection={
 'experiment': EXPERIMENT,
 'created_at': datetime.now(timezone.utc).isoformat(),
 'decision': 'wf2020_low_exposure_rally_floor_target_ready_for_official_replay_no_promotion',
 'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
 'direct_comparison_baseline': 'return_preserving_bull_participation_three_fold_checkpoint_metric_replay_v1',
 'reference_baseline': 'soxlbacktest_qlib_buysell_reference:risk_cap_35_15_05',
 'role': 'diagnose',
 'changed_layer': ['training_target_composition','execution_target','model_selection'],
 'target_window': 'wf_2020',
 'floor': FLOOR,
 'mom5_prev_threshold': MOM5_THRESHOLD,
 'target_fold_changed': TARGET_FOLD,
 'target_contract_path': rel(SIDE_CARS['target_contract']),
 'target_contract_rows': int(len(out)),
 'changed_rows': changed_rows,
 'target_eval_leakage_rows': int(audit['eval_leakage_rows'].sum()),
 'projected_mismatch_rows': int(audit['mismatch_rows'].sum()),
 'promotion_allowed': False,
 'actor_retention_allowed': False,
 'training_allowed_from_this_card': False,
 'checkpoint_replay_allowed_next': True,
 'next_mechanism': 'rerun_official_three_fold_checkpoint_replay_with_low_exposure_rally_floor_target',
 'closest_historical_success_case': 'wf2020_failed_fold_rally_gap_attribution_v1 found floor=0.30 late-rally diagnostic overlays can clear the return floor with MaxDD below 20%.',
 'closest_historical_failure_case': 'return_preserving replay still failed wf_2020 by 11.89pp with late-2020 position near 0.098.',
 'why_not_duplicate': 'This card writes a training-contract target floor from training rows only; it is not a post-hoc daily overlay.',
 'acceptance_criteria': {
    'wf2020_relative_return_pp_gte': -263.7287958891489,
    'wf2022_relative_return_pp_gte': 30.0,
    'wf2023_2024_relative_return_pp_gte': 30.0,
    'max_walkforward_drawdown_pct_lt': 20.0,
 },
 'sidecars': [path.name for path in SIDE_CARS.values()],
}
summary=dict(selection)
summary.update({
 'source_target_contract': rel(SOURCE_CONTRACT),
 'source_target_decision': source_summary.get('decision'),
 'attribution_decision': attribution.get('decision'),
 'fold_audit': audit.to_dict(orient='records'),
 'conclusion': [
    f'Applied floor {FLOOR:.2f} to {changed_rows} {TARGET_FOLD} training rows with mom5_prev >= {MOM5_THRESHOLD:.2f} and non-downtrend low exposure.',
    'The target has zero eval leakage and zero consumed-target mismatch rows.',
    'No actor is promoted; run official three-fold replay next.',
 ],
})
write_json(SIDE_CARS['selection'],selection)
write_json(SIDE_CARS['summary'],summary)
lines=[
 f'# {EXPERIMENT} conclusion','',f"Decision: `{selection['decision']}`.",'',
 f'- Changed rows: `{changed_rows}` in `{TARGET_FOLD}` only.',
 f'- Floor: `{FLOOR:.2f}` when `soxl_mom5_prev >= {MOM5_THRESHOLD:.2f}`, source target is below floor, and row is not downtrend.',
 f'- Eval leakage rows: `{int(audit["eval_leakage_rows"].sum())}`.',
 f'- Consumed-target mismatch rows: `{int(audit["mismatch_rows"].sum())}`.',
 '',
 'Promotion remains blocked. Next action: official three-fold checkpoint replay with this target contract.',
]
SIDE_CARS['conclusion'].write_text('\n'.join(lines).rstrip()+'\n',encoding='utf-8')
print(json.dumps(json_safe(summary),indent=2,allow_nan=False)[:5000])


{
  "experiment": "wf2020_low_exposure_rally_floor_target_v1",
  "created_at": "2026-06-14T08:21:23.957959+00:00",
  "decision": "wf2020_low_exposure_rally_floor_target_ready_for_official_replay_no_promotion",
  "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
  "direct_comparison_baseline": "return_preserving_bull_participation_three_fold_checkpoint_metric_replay_v1",
  "reference_baseline": "soxlbacktest_qlib_buysell_reference:risk_cap_35_15_05",
  "role": "diagnose",
  "changed_layer": [
    "training_target_composition",
    "execution_target",
    "model_selection"
  ],
  "target_window": "wf_2020",
  "floor": 0.3,
  "mom5_prev_threshold": -0.02,
  "target_fold_changed": "wf_2020_train",
  "target_contract_path": "runtime/model_selection/card/wf2020_low_exposure_rally_floor_target_v1/low_exposure_rally_floor_three_fold_target_contract.csv",
  "target_contract_rows": 3731,
  "changed_rows": 452,
  "target_eval_leakage_rows": 0,
  "projected_mismatch_rows": 0,
  "promo

## Conclusion

This notebook only writes the training-safe target contract. Actor retention remains blocked until the official three-fold replay is rerun and passes.